
# Same AGN, different viewing angle: Type 1 to Type 2 by inclination

The unified AGN model attributes the Type-1 vs Type-2 dichotomy to
geometry alone. Three inclinations of an identical disc + SKIRTOR
torus + broad-line region (Type 1, face-on, cos i = 0.95), torus
edge (intermediate, cos i = 0.5), and edge-on (Type 2, cos i = 0.1).
The broad UV bump and BLR lines vanish behind the torus at high
inclination; the mid-IR torus reprocessed emission stays.

Reference: Antonucci 1993, ARA&A, 31, 473 (unified model);
Urry & Padovani 1995, PASP, 107, 803.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

C_AA_PER_S = 2.998e18

INCLINATIONS = (
    ("Type 1 (face-on, cos i = 0.95)", 0.95, "#1f77b4"),
    ("Intermediate (cos i = 0.50)", 0.50, "#ff7f0e"),
    ("Type 2 (edge-on, cos i = 0.10)", 0.10, "#d62728"),
)
SFH = {"type": "const", "*": tengri.FIXED, "log_sfr": -10.0}
DUST = {"type": "two_component", "*": tengri.FIXED, "tau_diff": 0.0, "tau_bc": 0.0}

ssp = tengri.load_ssp()
fig, ax = plt.subplots(figsize=(7.5, 4.8))
for label, cos_inc, color in INCLINATIONS:
    model = tengri.SEDModel.build(
        ssp,
        sfh=SFH,
        dust=DUST,
        agn={
            "*": tengri.FIXED,
            "log_lbol": 12.5,
            "frac": 1.0,
            "cos_inc": cos_inc,
            "disc": {"type": "multicolor", "*": tengri.FIXED},
            "torus": {"type": "skirtor", "*": tengri.FIXED},
            "lines": {"type": "blr", "*": tengri.FIXED},
        },
        redshift=tengri.Fixed(0.0),
    )
    p = dict(model.spec.sample(jax.random.PRNGKey(0)))
    out = model.predict_rest_sed(p)
    wave_um = np.asarray(out.wavelength) * 1.0e-4
    nu_l_nu = C_AA_PER_S / np.asarray(out.wavelength) * np.asarray(out.sed)
    ax.loglog(wave_um, nu_l_nu, color=color, lw=1.8, label=label)

for wl_um, name in [(0.1216, r"Ly$\alpha$"), (0.6563, r"H$\alpha$"), (9.7, "silicate")]:
    ax.axvline(wl_um, color="0.6", ls=":", lw=0.7)
    ax.text(wl_um * 1.05, 1.5e41, name, fontsize=8, color="0.45", va="bottom", rotation=90)

ax.set(
    xlim=(1.0e-3, 100.0),
    ylim=(1.0e41, 1.0e46),
    xlabel=r"Rest-frame wavelength $\lambda$ [$\mu$m]",
    ylabel=r"$\nu L_\nu$  [erg s$^{-1}$]",
)
ax.legend(frameon=False, fontsize=9, loc="lower center")
fig.tight_layout()
plt.savefig("plot_agn_type12.png", dpi=150, bbox_inches="tight")